# Progress Review 2 — Train Spark MLlib Model

This notebook trains a Spark MLlib classifier on Gold training features.

Flow:

Gold features from MinIO → Spark MLlib Random Forest → evaluation metrics → saved model

The project proposal selected Spark MLlib for model training on lakehouse data.

In [1]:
import os
import pyspark
from pyspark import SparkContext

print("PySpark package version:", pyspark.__version__)
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

existing_sc = SparkContext._active_spark_context

if existing_sc is not None:
    try:
        if existing_sc._jvm is None or existing_sc._jsc is None:
            raise RuntimeError("JVM not available – context is dead")
        existing_sc.stop()
        print("Stopped existing SparkContext.")
    except Exception as e:
        print(f"Existing SparkContext is already dead ({e}); clearing reference.")
    finally:
        SparkContext._active_spark_context = None
        SparkContext._gateway = None
        SparkContext._jvm = None

print("SparkContext cleared – safe to create a new session.")

PySpark package version: 4.0.0
SPARK_HOME: /opt/conda/lib/python3.13/site-packages/pyspark
SparkContext cleared – safe to create a new session.


In [2]:
import pandas as pd
import s3fs

from deltalake import DeltaTable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [3]:
spark = (
    SparkSession.builder
    .appName("aviation-pr2-train-mllib")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "3g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "aviation-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4041")
    .config("spark.blockManager.port", "4042")
    .config("spark.ui.port", "4040")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark app ID:", spark.sparkContext.applicationId)

Spark version: 4.0.0
Spark app ID: app-20260425121414-0026


In [4]:
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT_INTERNAL", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    client_kwargs={"endpoint_url": MINIO_ENDPOINT},
)

storage_options = {
    "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
    "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
    "AWS_REGION": AWS_REGION,
    "AWS_ALLOW_HTTP": "true",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
}

In [5]:
gold_delta_uri = "s3://lakehouse/gold/training_features_delta"

try:
    dt = DeltaTable(gold_delta_uri, storage_options=storage_options)
    features_pdf = dt.to_pandas()
    print("Read Gold features from Delta:", gold_delta_uri)

except Exception as exc:
    print("Delta read failed, falling back to Parquet.")
    print("Error:", repr(exc))

    gold_parquet_path = "s3://lakehouse/gold/training_features_parquet/training_features.parquet"
    with fs.open(gold_parquet_path, "rb") as f:
        features_pdf = pd.read_parquet(f)

print("Gold shape:", features_pdf.shape)
display(features_pdf.head())

if features_pdf.empty:
    raise RuntimeError("Gold training features table is empty.")

Read Gold features from Delta: s3://lakehouse/gold/training_features_delta
Gold shape: (72, 17)


,event_id,airport,timestamp_utc,temperature_c,wind_speed_kts,wind_speed_max_3h,wind_speed_avg_3h,precipitation_mm,precipitation_sum_3h,surface_pressure_pa,cape_j_kg,cape_avg_3h,hour_of_day,day_of_week,month,label,label_source
0,JFK_2022-01-01T00:00:00,JFK,2022-01-01 00:00:00,9.424164,3.638606,3.638606,3.638606,0.005761,0.005761,101271.765625,12.700439,12.700439,0,7,1,0.0,PR2_QUANTILE_FALLBACK
1,JFK_2022-01-01T01:00:00,JFK,2022-01-01 01:00:00,9.465088,3.977354,3.977354,3.807980,0.010565,0.016326,101288.007812,21.743164,17.221802,1,7,1,0.0,PR2_QUANTILE_FALLBACK
2,JFK_2022-01-01T02:00:00,JFK,2022-01-01 02:00:00,9.402130,4.554652,4.554652,4.056870,0.012485,0.028811,101208.484375,23.470459,19.304688,2,7,1,0.0,PR2_QUANTILE_FALLBACK
3,JFK_2022-01-01T03:00:00,JFK,2022-01-01 03:00:00,9.392700,4.152695,4.554652,4.228234,0.001440,0.024490,101199.937500,17.780762,20.998128,3,7,1,0.0,PR2_QUANTILE_FALLBACK
4,JFK_2022-01-01T04:00:00,JFK,2022-01-01 04:00:00,9.112518,4.710645,4.710645,4.472664,0.000959,0.014884,101201.640625,14.834229,18.695150,4,7,1,0.0,PR2_QUANTILE_FALLBACK


In [6]:
training_df = spark.createDataFrame(features_pdf)

feature_cols = [
    "temperature_c",
    "wind_speed_kts",
    "wind_speed_max_3h",
    "wind_speed_avg_3h",
    "precipitation_mm",
    "precipitation_sum_3h",
    "surface_pressure_pa",
    "cape_j_kg",
    "cape_avg_3h",
    "hour_of_day",
    "day_of_week",
    "month",
]

for col_name in feature_cols:
    training_df = training_df.withColumn(col_name, F.col(col_name).cast("double"))

training_df = training_df.withColumn("label", F.col("label").cast("double"))

training_df.select(feature_cols + ["label"]).show(5, truncate=False)
training_df.groupBy("label").count().show()

+-----------------+-----------------+-----------------+------------------+---------------------+--------------------+-------------------+---------------+------------------+-----------+-----------+-----+-----+
|temperature_c    |wind_speed_kts   |wind_speed_max_3h|wind_speed_avg_3h |precipitation_mm     |precipitation_sum_3h|surface_pressure_pa|cape_j_kg      |cape_avg_3h       |hour_of_day|day_of_week|month|label|
+-----------------+-----------------+-----------------+------------------+---------------------+--------------------+-------------------+---------------+------------------+-----------+-----------+-----+-----+
|9.424163818359375|3.638605833053589|3.638605833053589|3.638605833053589 |0.00576116144657135  |0.00576116144657135 |101271.765625      |12.700439453125|12.700439453125   |0.0        |7.0        |1.0  |0.0  |
|9.465087890625   |3.977353811264038|3.977353811264038|3.8079798221588135|0.010564923286437988 |0.01632608473300934 |101288.0078125     |21.7431640625  |17.22180175

In [7]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
)

model_input_df = assembler.transform(training_df).select("features", "label")

train_df, test_df = model_input_df.randomSplit([0.8, 0.2], seed=42)

if train_df.select("label").distinct().count() < 2:
    print("Train split has only one class. Using full data for PR2 training demo.")
    train_df = model_input_df
    test_df = model_input_df

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

train_df.groupBy("label").count().show()

Train rows: 56
Test rows: 16
+-----+-----+
|label|count|
+-----+-----+
|  1.0|   14|
|  0.0|   42|
+-----+-----+



In [8]:
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=20,
    maxDepth=5,
    seed=42,
)

rf_model = rf.fit(train_df)

predictions = rf_model.transform(test_df)

predictions.select("label", "prediction", "probability").show(10, truncate=False)

+-----+----------+-----------+
|label|prediction|probability|
+-----+----------+-----------+
|0.0  |0.0       |[1.0,0.0]  |
|0.0  |0.0       |[0.95,0.05]|
|0.0  |0.0       |[0.95,0.05]|
|0.0  |0.0       |[0.95,0.05]|
|1.0  |1.0       |[0.25,0.75]|
|0.0  |0.0       |[1.0,0.0]  |
|0.0  |0.0       |[1.0,0.0]  |
|0.0  |0.0       |[1.0,0.0]  |
|1.0  |1.0       |[0.0,1.0]  |
|1.0  |1.0       |[0.0,1.0]  |
+-----+----------+-----------+
only showing top 10 rows


In [9]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy",
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1",
)

accuracy = accuracy_evaluator.evaluate(predictions)
f1 = f1_evaluator.evaluate(predictions)

print("Accuracy:", accuracy)
print("F1:", f1)

try:
    auc_evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    )
    auc = auc_evaluator.evaluate(predictions)
except Exception as exc:
    auc = None
    print("AUC unavailable:", repr(exc))

print("AUC:", auc)

Accuracy: 1.0
F1: 1.0
AUC: 1.0


In [10]:
model_path = "file:///spark-models/pr2_random_forest_model"


rf_model.write().overwrite().save(model_path)

print("Saved Spark MLlib model to:", model_path)

Saved Spark MLlib model to: file:///spark-models/pr2_random_forest_model


In [11]:
spark.stop()
print("Spark stopped cleanly.")

Spark stopped cleanly.
